# Simple BERTopic Pipeline (Hippocorpus)

1. Configure topic mode and load data.
2. Optional BERTopic-native LLM representation for topic names during modeling.
3. Fit BERTopic and export topic outputs.
4. Optional post-fit LLM suggestion and review workflow for topic names.
5. Label the full Hippocorpus with regex rules.
6. Append labels to reduced datasets by statement index.

In [15]:
from pathlib import Path
import os
import re
import pandas as pd
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

## Step 1: Configure topic fitting mode and load data
- Set `TOPIC_MODE` (`'hdbscan'` or `'kmeans'`).
- Configure paths and `APPEND_TARGETS`.
- Load Hippocorpus into `hip` and `docs`.

In [16]:
DATA_DIR = Path('../data')
HIPPOCORPUS_PATH = DATA_DIR / 'hippocorpus.csv'

# Topic fitting mode: 'hdbscan' (default) or 'kmeans'
TOPIC_MODE = 'hdbscan'
KMEANS_N_CLUSTERS = 30

OUTPUT_DOCS_PATH = DATA_DIR / ('output_k_means.csv' if TOPIC_MODE == 'kmeans' else 'output.csv')
OUTPUT_TOPICS_PATH = DATA_DIR / ('topic_info_k_means.csv' if TOPIC_MODE == 'kmeans' else 'topic_info.csv')
LABELED_HIPPOCORPUS_PATH = DATA_DIR / 'hippocorpus_labeled.csv'

# Add or remove targets here whenever you want to append labels to new files.
APPEND_TARGETS = [
    {'input': 'final.csv', 'index_col': 'os_id', 'output': 'final_labeled.csv'},
    {'input': 'final_deltamax.csv', 'index_col': 'os_id', 'output': 'final_deltamax_labeled.csv'},
    {'input': 'gemma4_raw.csv', 'index_col': 'statement_id', 'output': 'gemma4_raw_labeled.csv'},
    {'input': 'llama3.3_raw.csv', 'index_col': 'statement_id', 'output': 'llama3.3_raw_labeled.csv'},
    {'input': 'qwen3_raw.csv', 'index_col': 'statement_id', 'output': 'qwen3_raw_labeled.csv'},
    {'input': 'study1_final.csv', 'index_col': 'statement_id', 'output': 'study1_final_labeled.csv'},
    {'input': 'hippocorpus_test_truncated_80_100.csv', 'index_col': 'index', 'output': 'hippocorpus_test_truncated_80_100_labeled.csv'},
]

hip = pd.read_csv(HIPPOCORPUS_PATH, sep=',')
docs = hip['text'].astype(str).tolist()
print(f'Hippocorpus rows: {len(hip):,}')

Hippocorpus rows: 5,047


## Step 2 (Optional): Use BERTopic LLM representation model

This is the BERTopic-native approach to naming topics during modeling.

- Set `USE_BERTOPIC_LLM_REPRESENTATION = True`.
- Export `OPENAI_API_KEY` before running.
- If disabled or unavailable, the pipeline falls back to `KeyBERTInspired`.

In [ ]:
USE_BERTOPIC_LLM_REPRESENTATION = False
BERTOPIC_LLM_MODEL = 'gpt-4.1-mini'

REPRESENTATION_MODEL = KeyBERTInspired()

if USE_BERTOPIC_LLM_REPRESENTATION:
    try:
        from openai import OpenAI
        from bertopic.representation import OpenAI as BERTopicOpenAI
    except ImportError as exc:
        raise ImportError(
            'Install openai to use BERTopic OpenAI representation: pip install openai'
        ) from exc

    api_key = os.getenv('OPENAI_API_KEY', '')
    if not api_key:
        raise ValueError('Missing OPENAI_API_KEY environment variable.')

    client = OpenAI(api_key=api_key)
    prompt = (
        'You are an expert topic modeler. Given topic keywords, generate one concise '\
        'human-readable topic label (2-3 words). Return only the label text.'
    )
    REPRESENTATION_MODEL = BERTopicOpenAI(
        client,
        model=BERTOPIC_LLM_MODEL,
        prompt=prompt,
        chat=True,
    )
    print(f'Using BERTopic OpenAI representation model: {BERTOPIC_LLM_MODEL}')
else:
    print('Using default BERTopic representation model: KeyBERTInspired')

## Step 3: Fit topics

In [ ]:
representation_model = REPRESENTATION_MODEL if 'REPRESENTATION_MODEL' in globals() else KeyBERTInspired()

if TOPIC_MODE == 'kmeans':
    cluster_model = KMeans(n_clusters=KMEANS_N_CLUSTERS, random_state=42)
    vectorizer_model = CountVectorizer(stop_words='english')

    topic_model = BERTopic(
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
    )
    topics, probs = topic_model.fit_transform(docs)

    # Old kmeans pipeline: no outlier reduction step
    hip['topic'] = topics
else:
    topic_model = BERTopic(representation_model=representation_model)
    topics, probs = topic_model.fit_transform(docs)

    # Old hdbscan pipeline: reduce outliers then update topic representations
    reduced_topics = topic_model.reduce_outliers(docs, topics)
    hip['topic'] = reduced_topics
    topic_model.update_topics(docs, topics=reduced_topics)

hip.to_csv(OUTPUT_DOCS_PATH, index=False)
topic_info = topic_model.get_topic_info()
topic_info.to_csv(OUTPUT_TOPICS_PATH, index=False)

print(f'Saved: {OUTPUT_DOCS_PATH}')
print(f'Saved: {OUTPUT_TOPICS_PATH}')
topic_info.head(10)

## Step 4 (Optional Alternative): Post-fit LLM topic name suggestions

Use this step only if you want a separate review/apply loop after fitting topics.

- Set `USE_LLM_TOPIC_NAMES = True`.
- Export your key before running: `export OPENAI_API_KEY=...`
- Review `data/topic_name_suggestions.csv` before applying any labels.

The next code cell only creates suggestions; it does not change your dataset labels automatically.

In [ ]:
import json
import os
from urllib import error, request

# Set to True to request suggested topic names from an OpenAI-compatible API.
USE_LLM_TOPIC_NAMES = False
LLM_API_KEY = os.getenv('OPENAI_API_KEY', '')
LLM_BASE_URL = 'https://api.openai.com/v1'
LLM_MODEL = 'gpt-4.1-mini'
LLM_OUTPUT_PATH = DATA_DIR / 'topic_name_suggestions.csv'


def build_topic_payload(topic_info_df, max_topics=40):
    keep = topic_info_df[topic_info_df['Topic'] != -1].head(max_topics).copy()
    payload = []
    for row in keep.itertuples(index=False):
        representation = row.Representation
        if not isinstance(representation, list):
            representation = [tok.strip() for tok in str(representation).strip('[]').replace("'", '').split(',') if tok.strip()]
        payload.append(
            {
                'topic_id': int(row.Topic),
                'count': int(row.Count),
                'current_name': str(row.Name),
                'keywords': representation[:10],
            }
        )
    return payload


if not USE_LLM_TOPIC_NAMES:
    print('LLM naming disabled. Set USE_LLM_TOPIC_NAMES = True to generate suggestions.')
else:
    if not LLM_API_KEY:
        raise ValueError('Missing OPENAI_API_KEY environment variable.')

    topics_payload = build_topic_payload(topic_info)
    prompt = {
        'task': 'Suggest short, human-readable labels for BERTopic topics.',
        'rules': [
            'Label length: 2-5 words.',
            'Be specific and concrete.',
            'Avoid numbering in the label text.',
            'Return valid JSON only.',
        ],
        'return_format': {
            'topic_names': {
                '<topic_id>': '<suggested_label>'
            }
        },
        'topics': topics_payload,
    }

    body = {
        'model': LLM_MODEL,
        'temperature': 0.2,
        'messages': [
            {'role': 'system', 'content': 'You rename topic-model clusters for analytics reporting.'},
            {'role': 'user', 'content': json.dumps(prompt)},
        ],
    }

    req = request.Request(
        f"{LLM_BASE_URL.rstrip('/')}/chat/completions",
        data=json.dumps(body).encode('utf-8'),
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {LLM_API_KEY}',
        },
        method='POST',
    )

    try:
        with request.urlopen(req, timeout=90) as resp:
            raw = resp.read().decode('utf-8')
    except error.HTTPError as exc:
        detail = exc.read().decode('utf-8', errors='ignore')
        raise RuntimeError(f'LLM API error {exc.code}: {detail}') from exc

    response_json = json.loads(raw)
    content = response_json['choices'][0]['message']['content']

    try:
        parsed = json.loads(content)
    except json.JSONDecodeError:
        start = content.find('{')
        end = content.rfind('}')
        if start == -1 or end == -1:
            raise ValueError('Model response was not valid JSON.')
        parsed = json.loads(content[start : end + 1])

    mapping = parsed.get('topic_names', parsed)

    suggestions = []
    for topic_id, label in mapping.items():
        try:
            topic_int = int(topic_id)
        except (TypeError, ValueError):
            continue
        suggestions.append({'Topic': topic_int, 'suggested_label': str(label).strip()})

    suggestion_df = pd.DataFrame(suggestions).drop_duplicates(subset='Topic')
    suggestion_df = topic_info[['Topic', 'Name', 'Count']].merge(suggestion_df, on='Topic', how='left')
    suggestion_df.to_csv(LLM_OUTPUT_PATH, index=False)

    print(f'Saved: {LLM_OUTPUT_PATH}')
    suggestion_df.head(20)

In [ ]:
# Optional: apply reviewed topic names back to topic_info.
# Set APPLY_LLM_NAMES = True only after checking LLM_OUTPUT_PATH.
APPLY_LLM_NAMES = False
FINAL_TOPIC_INFO_PATH = DATA_DIR / ('topic_info_k_means_named.csv' if TOPIC_MODE == 'kmeans' else 'topic_info_named.csv')

if not APPLY_LLM_NAMES:
    print('Apply step disabled. Set APPLY_LLM_NAMES = True to write named topic info.')
else:
    if not LLM_OUTPUT_PATH.exists():
        raise FileNotFoundError(f'Missing suggestion file: {LLM_OUTPUT_PATH}')

    reviewed = pd.read_csv(LLM_OUTPUT_PATH)
    required_cols = {'Topic', 'suggested_label'}
    missing_cols = required_cols.difference(reviewed.columns)
    if missing_cols:
        raise ValueError(f'Suggestion file is missing columns: {sorted(missing_cols)}')

    reviewed = reviewed[['Topic', 'suggested_label']].copy()
    reviewed['Topic'] = pd.to_numeric(reviewed['Topic'], errors='coerce').astype('Int64')
    reviewed['suggested_label'] = reviewed['suggested_label'].astype(str).str.strip()
    reviewed = reviewed.dropna(subset=['Topic'])
    reviewed = reviewed[reviewed['suggested_label'].ne('')]
    reviewed = reviewed[reviewed['suggested_label'].str.lower().ne('nan')]
    reviewed = reviewed.drop_duplicates(subset='Topic', keep='last')

    topic_info_named = topic_info.copy()
    topic_info_named = topic_info_named.merge(
        reviewed.rename(columns={'suggested_label': 'llm_label'}),
        on='Topic',
        how='left',
    )
    topic_info_named['final_name'] = topic_info_named['llm_label'].fillna(topic_info_named['Name'])

    topic_info_named.to_csv(FINAL_TOPIC_INFO_PATH, index=False)
    print(f'Saved: {FINAL_TOPIC_INFO_PATH}')
    topic_info_named[['Topic', 'Name', 'llm_label', 'final_name', 'Count']].head(20)

## Step 5: Create regex rules from inspected topics and label full Hippocorpus
Edit only `RULES` below. First match wins.

In [86]:
RULES = [
    (1, 'Funeral/Death', re.compile(
        r'\b(funeral|wake(?= service| ceremony|\s+for\b)|\bburial|cremat(?:e|ion|ed)|cemeter(?:y|ies)|(?:he|she|they|someone|my\s+\w+)\s+died|passed away|grief|mourning|eulog(?:y|ies))\b',
        re.I,
    )),
    (2, 'NaturalDisaster/Weather', re.compile(
        r'\b(hurricane|tornado|earthquake|aftershock|wildfire|'
        r'evacuat(?:e|ed|ion)|'
        r'flash\s+flood(?:ed|ing)?|flood(?:ed|ing)\s+(?:the\s+)?(?:town|street|city|neighborhood|area|region)|'
        r'house\s+flooded|'
        r'water\s+damage\s+(?:from|caused\s+by)\s+(?:the\s+)?(?:hurricane|flood|storm|rain|hail)|'
        r'storm\s+surge|blizzard|ice\s+storm|hailstorm)\b',
        re.I,
    )),
    (3, 'Hospital/Injury', re.compile(
        r'\b(hospital(?:ized)?|(?:my\s+)?doctor|dr\.(?=\s+[A-Z])|'
        r'(?:registered\s+)?nurse(?=\s+(?:said|told|came|checked|practitioner))|'
        r'clinic|emergency\s+room|ambulance|surgery|chemo(?:therapy)?|tumou?r|'
        r'(?:physical\s+)?therapy|medication|diagnos(?:is|ed)|'
        r'injur(?:y|ed)|wound(?:ed)?|stitch(?:es)?|'
        r'(?:had\s+(?:a\s+)?|with\s+(?:a\s+)?)broken\s+(?:arm|leg|bone|wrist|ankle|rib|collarbone|toe|finger|jaw|nose|hip)|'
        r'fracture(?:d)?|sprain(?:ed)?|concussion|'
        r'(?:got\s+(?:a\s+|an\s+)?|developed\s+(?:a\s+|an\s+)?|treated\s+for\s+(?:a\s+|an\s+)?)infection)\b',
        re.I,
    )),
    (4, 'Gambling', re.compile(
        r'\b(casino|gambl(?:e|ing|er)|'
        r'(?:placed|made|setting)\s+(?:a\s+)?wager(?:\s+on)?|'
        r'poker|blackjack|roulette|'
        r'(?:won\s+the\s+)?lottery|winnings?|jackpot|'
        r'won\s+(?:big|a\s+lot\s+of\s+)?money)\b',
        re.I,
    )),
    (5, 'Pets', re.compile(
        r'\b((?:my\s+)?dog|puppy|(?:my\s+)?cat|kitten|(?:my\s+)?pets?|veterinarian|animal\s+hospital|'
        r'on\s+(?:a\s+|the\s+)leash|'
        r'adopt(?:ed|ing|ion)\s+(?:a\s+)?(?:dog|cat|pet|puppy|kitten)|'
        r'walking\s+(?:my\s+|the\s+)?dog|'
        r'took\s+(?:my\s+|the\s+)?(?:dog|cat)\s+to\s+the\s+vet)\b',
        re.I,
    )),
    (6, 'Concert/Music', re.compile(
        r'\b(concert|(?:played\s+a\s+)?gig|live\s+music|'
        r'(?:my\s+)?favorite\s+band|(?:the\s+)?band\s+(?:played|was|is|came|performed)|'
        r'music\s+festival|'
        r'setlist)\b',
        re.I,
    )),
    (7, 'Vacation/Trip', re.compile(
        r'\b(vacation|holiday(?!\s+(?:party|bonus|pay))|road\s+trip|travel(?:ed|ing|ler)?|(?:booked?\s+a?\s+)?flight|airport|hotel|resort|airbnb|beach(?=\s+(?:vacation|trip|house))?|hiking\s+trail|went\s+camping|'
        r'(?:rented?\s+(?:a\s+)?|stayed\s+in\s+(?:a\s+)?)cabin|'
        r'kayak(?:ing)?|rafting|national\s+park|skiing|snowboard(?:ing)?|surfing|paris|london|rome|new\s+york)\b',
        re.I,
    )),
    (8, 'Birthday', re.compile(
        r'\b((?:my\s+|his\s+|her\s+|their\s+)birthday|bday|'
        r'(?:my\s+|his\s+|her\s+|their\s+)\d{1,3}(?:st|nd|rd|th)?\s+birthday|'
        r'(?:hosted|threw)\s+(?:a\s+)?birthday(?:\s+party)?|'
        r'(?:good|great)\s+birthday|had\s+a\s+(?:good|great)\s+birthday|'
        r'birthday\s+(?:cake|party|dinner|celebration|candles|present(?:s)?|gift(?:s)?)|'
        r'blew\s+out\s+(?:the\s+)?candles)\b',
        re.I,
    )),
    (9, 'Children', re.compile(
        r'\b(newborn|toddler|'
        r'(?:my\s+)?child(?:ren)?|'
        r'(?:my\s+(?:kids?|children)|kids?\s+(?:are|were|have|had|need|started|love|go|went))|'
        r'pregnan(?:t|cy)|'
        r'expecting\s+(?:a\s+baby|our\s+(?:first|second|third))|'
        r'(?:she\s+)?went\s+into\s+labor|'
        r'delivery\s+room|'
        r'gave\s+birth|'
        r'(?:our|the|my)\s+baby(?:\s+(?:was\s+born|is\s+due|arrived|shower))?|'
        r'ultrasound|due\s+date|diapers?|daycare|crib|stroller)\b',
        re.I,
    )),
    (10, 'Work', re.compile(
        r'\b((?:my\s+)?(?:new\s+)?job|'
        r'at\s+work|'
        r'my\s+company|my\s+boss|my\s+manager|'
        r'coworkers?|colleagues?|'
        r'(?:at\s+the\s+)?office|'
        r'(?:work\s+)?shift(?:\s+at\s+work)?|'
        r'(?:got\s+a\s+)?promotion|(?:got\s+)?promoted|'
        r'(?:got|received|earned|given)\s+a\s+raise|'
        r'job\s+interview|'
        r'my\s+resum[eé]|\bcv\b|'
        r'(?:got\s+)?hired|(?:got\s+)?fired|laid\s+off|'
        r'quit\s+(?:my\s+)?(?:job|position)|'
        r'resign(?:ed|ation)?)\b',
        re.I,
    )),
    (11, 'School', re.compile(
        r'\b(university|college|'
        r'(?:high\s+|middle\s+|elementary\s+)?school|'
        r'campus|'
        r'my\s+class(?:es)?|'
        r'lecture(?:s)?|'
        r'(?:my\s+)?teacher|professor|homework|assignment(?:s)?|'
        r'(?:final\s+)?exam(?:s)?|'
        r'(?:school|course|university)\s+finals?|finals?\s+(?:at\s+school|week)|'
        r'graduat(?:e|ed|ion)|(?:got\s+my\s+)?degree|diploma|'
        r'graduation\s+ceremony)\b',
        re.I,
    )),
    (12, 'Home/Moving', re.compile(
        r'\b((?:bought|rented)\s+(?:a\s+)?(?:house|apartment|flat|condo)|'
        r'my\s+flat|condo|roommate(?:s)?|mortgage|'
        r'signed\s+(?:a\s+)?lease|'
        r'landlord|tenant|'
        r'moved\s+in(?:to)?|just\s+moved|'
        r'packing\s+(?:boxes|up\s+my\s+(?:stuff|things|apartment|house|flat))|'
        r'moving\s+(?:boxes|day|truck|company|houses|apartment)|'
        r'new\s+(?:place|apartment|house|home|flat))\b',
        re.I,
    )),
    (13, 'Relationships', re.compile(
        r'\b((?:my\s+)?(?:wife|husband|fianc(?:e|ée)|boyfriend|girlfriend)|'
        r'went\s+on\s+a\s+date|date\s+night|'
        r'(?:am\s+|I\'m\s+|been\s+)?single|'
        r'broke?\s+up|break\s*up|'
        r'asked\s+(?:her|him|them)\s+(?:to\s+marry|out)|'
        r'wedding|(?:our\s+|the\s+|my\s+)?engagement|propos(?:ed?|al)|'
        r'engagement\s+ring|fell\s+in\s+love)\b',
        re.I,
    )),
    (14, 'Sports', re.compile(
        r'\b(playoffs?|tournament|match|'
        r'(?:my\s+|the\s+)?(?:sports?\s+)?coach(?!\s+(?:bus|seat|class))|'
        r'(?:sports?\s+practice|team\s+practice)|'
        r'(?:ran|finished)\s+(?:a\s+)?(?:race|5k|10k|half\s+marathon|marathon|triathlon)|'
        r'marathon|triathlon|'
        r'(?:hit\s+the\s+)?gym|'
        r'baseball|basketball|soccer|football|tennis|hockey|'
        r'swim(?:ming)\s+(?:team|race|meet)|'
        r'cycling|went\s+running|'
        r'(?:lifting\s+)?weights?)\b',
        re.I,
    )),
    (15, 'Family', re.compile(
        r'\b((?:my\s+)?(?:mom|mother|mum|dad|father)|'
        r'(?:my\s+)?(?:step(?:mom|mother|dad|father)|parents?)|'
        r'(?:my\s+)?(?:sister|brother|siblings?|daughter|son)|'
        r'(?:my\s+)?(?:grandma|grandpa|grandmother|grandfather|aunt|uncle|cousins?|niece|nephew)|'
        r'family\s+reunion)\b',
        re.I,
    )),
    (16, 'Writing', re.compile(
        r'\b((?:kept?\s+(?:a\s+)?|keep(?:ing)?\s+(?:a\s+)?)diary|dear\s+diary|'
        r'(?:start(?:ed|ing)?\s+(?:a\s+)?|kept?\s+(?:a\s+)?|keep(?:ing)?\s+(?:a\s+)?)journal(?:ing)?|'
        r'(?:wrote|writing|finished)\s+(?:a\s+)?(?:story|poem|novel|book|chapter|manuscript)|'
        r'publish(?:ed|ing)\s+(?:a\s+)?(?:book|novel|poem|manuscript|chapter)|'
        r'chapter(?=\s+(?:\d+|of|in))|'
        r'manuscript|poems?|poetry|'
        r'went\s+to\s+(?:the\s+)?library)\b',
        re.I,
    )),
]

def label_text(text):
    if pd.isna(text):
        return 0, 'Other'
    for topic_id, topic_label, pattern in RULES:
        if pattern.search(str(text)):
            return topic_id, topic_label
    return 0, 'Other'

In [87]:
# Optional rule-priority override (first match still wins).
RULE_ORDER = [
    'Funeral/Death',
    'NaturalDisaster/Weather',
    'Money/Gambling',
    'Concert/Music',
    'Vacation/Trip',
    'Birthday',
    'Hospital/Injury',
    'Children',
    'Work',
    'School',
    'Home/Moving',
    'Relationships',
    'Sports',
    'Family',
    'Pets',
    'Writing',
]

priority = {label: idx for idx, label in enumerate(RULE_ORDER)}
RULES = sorted(RULES, key=lambda rule: priority.get(rule[1], len(priority)))
print('Applied RULE_ORDER priority. Pets now runs after Family in matching.')

Applied RULE_ORDER priority. Pets now runs after Family in matching.


In [88]:
hip = pd.read_csv(HIPPOCORPUS_PATH)
hip[['topic_id', 'topic_label']] = hip['text'].apply(lambda x: pd.Series(label_text(x)))
hip.to_csv(LABELED_HIPPOCORPUS_PATH, index=False)

print(f'Saved: {LABELED_HIPPOCORPUS_PATH}')
hip['topic_label'].value_counts(dropna=False).head(20)

Saved: ../data/hippocorpus_labeled.csv


topic_label
Vacation/Trip              853
Hospital/Injury            697
Work                       589
Children                   507
Relationships              467
Other                      403
Funeral/Death              384
School                     333
Family                     254
Birthday                   188
Concert/Music               91
Home/Moving                 83
Pets                        71
Sports                      69
NaturalDisaster/Weather     29
Gambling                    15
Writing                     14
Name: count, dtype: int64

In [46]:
# Quick regex audit: distribution, overlap, and sample hits
from textwrap import shorten

N_OVERLAP_PREVIEW = 20
N_SAMPLES_PER_TOPIC = 3
PREVIEW_WIDTH = 180

if 'hip' not in globals() or not {'text', 'topic_id', 'topic_label'}.issubset(hip.columns):
    if LABELED_HIPPOCORPUS_PATH.exists():
        hip = pd.read_csv(LABELED_HIPPOCORPUS_PATH)
    else:
        hip = pd.read_csv(HIPPOCORPUS_PATH)
        hip[['topic_id', 'topic_label']] = hip['text'].apply(lambda x: pd.Series(label_text(x)))

topic_distribution = (
    hip['topic_label']
    .value_counts(dropna=False)
    .rename_axis('topic_label')
    .reset_index(name='count')
)
topic_distribution['pct'] = (topic_distribution['count'] / len(hip) * 100).round(2)

print('Topic distribution:')
display(topic_distribution)

texts = hip['text'].fillna('').astype(str)
rule_hits = pd.DataFrame(
    {topic_label: texts.str.contains(pattern, na=False) for _, topic_label, pattern in RULES}
 )
rule_hits_count = rule_hits.sum(axis=1)
multi_hit_mask = rule_hits_count > 1

ambiguous_rows = hip.loc[multi_hit_mask, ['text', 'topic_label']].copy()
if not ambiguous_rows.empty:
    ambiguous_rows['matched_topics'] = rule_hits.loc[multi_hit_mask].apply(
        lambda row: [col for col, matched in row.items() if matched], axis=1
    )
    ambiguous_rows['text_preview'] = ambiguous_rows['text'].astype(str).apply(
        lambda s: shorten(s.replace('\n', ' '), width=PREVIEW_WIDTH, placeholder=' ...')
    )

    print(f'Potential overlap rows (matched >1 rule): {len(ambiguous_rows):,}')
    display(ambiguous_rows[['topic_label', 'matched_topics', 'text_preview']].head(N_OVERLAP_PREVIEW))

    print('\nFull text for overlap preview rows:')
    for i, row in enumerate(ambiguous_rows.head(N_OVERLAP_PREVIEW).itertuples(index=False), start=1):
        print(f"\n[{i}] assigned={row.topic_label} | matched={', '.join(row.matched_topics)}")
        print(row.text)
else:
    print('Potential overlap rows (matched >1 rule): 0')

print('\nSample texts per assigned topic:')
sample_parts = []
for _, group in hip[['topic_label', 'text']].dropna(subset=['text']).groupby('topic_label'):
    sample_parts.append(group.sample(min(N_SAMPLES_PER_TOPIC, len(group)), random_state=42))

samples = pd.concat(sample_parts, ignore_index=True) if sample_parts else pd.DataFrame(columns=['topic_label', 'text'])
samples['text_preview'] = samples['text'].astype(str).apply(
    lambda s: shorten(s.replace('\n', ' '), width=PREVIEW_WIDTH, placeholder=' ...')
)
display(samples[['topic_label', 'text_preview']].head(60))

Topic distribution:


,topic_label,count,pct
0,Work,830,16.45
1,Vacation/Trip,788,15.61
2,Hospital/Injury,636,12.60
3,Children,477,9.45
4,Funeral/Death,384,7.61
5,Relationships,365,7.23
6,Pets,320,6.34
7,Other,309,6.12
8,School,240,4.76
9,Family,180,3.57


Potential overlap rows (matched >1 rule): 3,658


,topic_label,matched_topics,text_preview
0,Pets,"[Pets, Relationships, Sports, Family]",Rex needs time to learn and that's exactly wha...
1,Children,"[Children, Home/Moving, Relationships, Family]",About three weeks ago my best friend gets kick...
2,Funeral/Death,"[Funeral/Death, Work, Relationships, Family]","Two weeks ago, we buried my mother in law. It ..."
3,Hospital/Injury,"[Hospital/Injury, Work, Relationships]",So today my thoughts are I can not let this un...
4,Children,"[Children, School, Relationships, Family]",My son started college. We loaded up both of o...
5,Hospital/Injury,"[Hospital/Injury, Relationships, Sports]","My wife had always been a healthy person, alth..."
6,Hospital/Injury,"[Hospital/Injury, Sports, Family]","It's been a crazy year. Reflecting back, I fee..."
8,Work,"[Work, School, Relationships, Family]","Four months ago, I had a wild time going to a ..."
10,Work,"[Work, Family]",I recently hosted a bridal shower for my best ...
13,Vacation/Trip,"[Vacation/Trip, Relationships, Family]",A couple months ago my younger brother got mar...



Full text for overlap preview rows:

[1] assigned=Pets | matched=Pets, Relationships, Sports, Family
Rex needs time to learn and that's exactly what will make him feel part of the family.  My husband and i brought in a dog trainer and Rex is now so promising. He has learnt quite a lot and i am impressed on the speed at which he is learning. The last time we were at a park with our daughter and Rex helped us find her when we lost track of her as she rode her bike in the park. We heard Rex barking from a distant and we went running towards the direction of the sound. We were happy to find Rex and Stacy together under a tree.  On asking Stacy why she didn't come back, she announced that her bike had spoiled and could not move. She trusted that somehow we would go looking for her. It was such a good moment and we loved Rex even more. We have even bought him a new collar. It has his name written on it.  Rex is so promising.   There was this memorable night when burglars came home and were 

,topic_label,text_preview
0,Birthday,"Not so long back, about three weeks ago, my hu..."
1,Birthday,"Dear Diary, Today is my sisters 20th birthday...."
2,Birthday,I would like to share some of my unforgettable...
3,Children,My husband and I got married on a bright sunny...
4,Children,"Three months ago, my best friend got married. ..."
5,Children,I was extremely excited when I became a father...
6,Concert/Music,I am very glad to share my most unforgettable ...
7,Concert/Music,I was ecstatic when my childhood friend called...
8,Concert/Music,I was recently at an outdoor concert with my d...
9,Family,"A week ago, I learned there is no treatment fo..."


## Step 6: Append labels to reduced datasets by statement index

In [94]:
hip_labeled = pd.read_csv(LABELED_HIPPOCORPUS_PATH)

# Statement index alignment in the labeled Hippocorpus
hip_labeled['index'] = hip_labeled['index'].astype(str)
labels = hip_labeled[['index', 'topic_id', 'topic_label']].drop_duplicates(subset='index')

for target in APPEND_TARGETS:
    input_path = DATA_DIR / target['input']
    output_path = DATA_DIR / target['output']
    index_col = target['index_col']

    reduced = pd.read_csv(input_path)
    if index_col not in reduced.columns:
        print(f"Skipping {input_path}: missing join column '{index_col}'")
        continue

    reduced[index_col] = reduced[index_col].astype(str)

    merged = reduced.merge(labels, left_on=index_col, right_on='index', how='left')
    # Keep source index column when the join key itself is named 'index'.
    if index_col != 'index':
        merged = merged.drop(columns=['index'])

    missing = merged['topic_label'].isna().sum()

    merged.to_csv(output_path, index=False)
    print(f'Saved: {output_path} | Missing labels: {missing}')

Saved: ../data/final_labeled.csv | Missing labels: 0
Saved: ../data/final_deltamax_labeled.csv | Missing labels: 0
Saved: ../data/gemma4_raw_labeled.csv | Missing labels: 0
Saved: ../data/llama3.3_raw_labeled.csv | Missing labels: 0
Saved: ../data/qwen3_raw_labeled.csv | Missing labels: 0
Saved: ../data/study1_final_labeled.csv | Missing labels: 0
Saved: ../data/hippocorpus_test_truncated_80_100_labeled.csv | Missing labels: 0


Outputs:
- data/output.csv (or data/output_k_means.csv)
- data/topic_info.csv (or data/topic_info_k_means.csv)
- data/topic_name_suggestions.csv (optional LLM step)
- data/topic_info_named.csv (or data/topic_info_k_means_named.csv, optional apply step)
- data/hippocorpus_labeled.csv
- one labeled output per APPEND_TARGETS entry

## Step 7: Topic statistics overview for labeled CSVs

In [90]:
labeled_outputs = sorted({target['output'] for target in APPEND_TARGETS} | {'hippocorpus_labeled.csv'})

all_stats = []

for output_name in labeled_outputs:
    output_path = DATA_DIR / output_name
    if not output_path.exists():
        print(f'Skipping missing file: {output_path}')
        continue

    df = pd.read_csv(output_path)
    required_cols = {'topic_id', 'topic_label'}
    missing_cols = required_cols.difference(df.columns)
    if missing_cols:
        print(f"Skipping {output_name}: missing columns {sorted(missing_cols)}")
        continue

    stats = (
        df.groupby(['topic_id', 'topic_label'], dropna=False)
        .size()
        .rename('count')
        .reset_index()
    )
    stats['dataset'] = output_name
    stats['pct'] = (stats['count'] / len(df) * 100).round(2) if len(df) else 0.0
    all_stats.append(stats)

if all_stats:
    topic_stats_overview = pd.concat(all_stats, ignore_index=True)
    topic_stats_overview = topic_stats_overview[['dataset', 'topic_id', 'topic_label', 'count', 'pct']]
    topic_stats_overview = topic_stats_overview.sort_values(['dataset', 'count'], ascending=[True, False])

    overview_path = DATA_DIR / 'topic_stats_overview.csv'
    topic_stats_overview.to_csv(overview_path, index=False)

    print(f'Saved: {overview_path}')
    display(topic_stats_overview.head(50))

    dataset_totals = (
        topic_stats_overview.groupby('dataset', as_index=False)['count']
        .sum()
        .rename(columns={'count': 'rows'})
    )
    display(dataset_totals)
else:
    print('No labeled CSVs with topic columns were found.')

Saved: ../data/topic_stats_overview.csv


,dataset,topic_id,topic_label,count,pct
3,final_deltamax_labeled.csv,3,Hospital/Injury,70,17.33
6,final_deltamax_labeled.csv,7,Vacation/Trip,60,14.85
9,final_deltamax_labeled.csv,10,Work,60,14.85
8,final_deltamax_labeled.csv,9,Children,38,9.41
12,final_deltamax_labeled.csv,13,Relationships,30,7.43
1,final_deltamax_labeled.csv,1,Funeral/Death,28,6.93
0,final_deltamax_labeled.csv,0,Other,26,6.44
14,final_deltamax_labeled.csv,15,Family,22,5.45
10,final_deltamax_labeled.csv,11,School,18,4.46
7,final_deltamax_labeled.csv,8,Birthday,14,3.47


,dataset,rows
0,final_deltamax_labeled.csv,404
1,final_labeled.csv,2105
2,gemma4_raw_labeled.csv,330
3,hippocorpus_labeled.csv,5047
4,hippocorpus_test_truncated_80_100_labeled.csv,262
5,llama3.3_raw_labeled.csv,330
6,qwen3_raw_labeled.csv,330
7,study1_final_labeled.csv,330


In [91]:
# Deep dive for one labeled output
TARGET_OUTPUT = 'hippocorpus_test_truncated_80_100_labeled.csv'  # e.g., 'final_labeled.csv'
FILTER_TOPIC_LABEL = None  # e.g., 'Hospital/Injury'
TOP_N_TOPICS = 30
TOP_N_TEXTS_PER_TOPIC = 5
TEXT_PREVIEW_WIDTH = 220
SHOW_FULL_TEXT = True
FULL_TEXT_ROWS = 80

target_path = DATA_DIR / TARGET_OUTPUT
if not target_path.exists():
    raise FileNotFoundError(f'Missing file: {target_path}')

target_df = pd.read_csv(target_path)
required_cols = {'topic_id', 'topic_label'}
missing_cols = required_cols.difference(target_df.columns)
if missing_cols:
    raise ValueError(f"{TARGET_OUTPUT} is missing columns: {sorted(missing_cols)}")

topic_dist_target = (
    target_df.groupby(['topic_id', 'topic_label'], dropna=False)
    .size()
    .rename('count')
    .reset_index()
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)
topic_dist_target['pct'] = (topic_dist_target['count'] / len(target_df) * 100).round(2) if len(target_df) else 0.0

if FILTER_TOPIC_LABEL is not None:
    topic_dist_target = topic_dist_target[topic_dist_target['topic_label'] == FILTER_TOPIC_LABEL].reset_index(drop=True)

print(f'Deep dive dataset: {TARGET_OUTPUT} | Rows: {len(target_df):,}')
display(topic_dist_target.head(TOP_N_TOPICS))

if 'text' in target_df.columns:
    from textwrap import shorten
    if FILTER_TOPIC_LABEL is None:
        top_labels = topic_dist_target['topic_label'].head(TOP_N_TOPICS).tolist()
        subset = target_df[target_df['topic_label'].isin(top_labels)].copy()
    else:
        subset = target_df[target_df['topic_label'] == FILTER_TOPIC_LABEL].copy()

    sample_parts = []
    for _, grp in subset[['topic_label', 'text']].dropna(subset=['text']).groupby('topic_label'):
        sample_parts.append(grp.sample(min(TOP_N_TEXTS_PER_TOPIC, len(grp)), random_state=42))

    if sample_parts:
        sample_texts = pd.concat(sample_parts, ignore_index=True)
        sample_texts['text_preview'] = sample_texts['text'].astype(str).apply(
            lambda s: shorten(s.replace('\n', ' '), width=TEXT_PREVIEW_WIDTH, placeholder=' ...')
        )
        display(sample_texts[['topic_label', 'text_preview']])

        if SHOW_FULL_TEXT:
            print('\nFull text samples:')
            for i, row in enumerate(sample_texts.head(FULL_TEXT_ROWS).itertuples(index=False), start=1):
                print(f"\n[{i}] topic={row.topic_label}")
                print(row.text)
    else:
        print('No non-empty text rows available for sampling in this filter.')
else:
    print(f"{TARGET_OUTPUT} has no 'text' column, so text examples are not available.")

Deep dive dataset: hippocorpus_test_truncated_80_100_labeled.csv | Rows: 262


,topic_id,topic_label,count,pct
0,7,Vacation/Trip,46,17.56
1,3,Hospital/Injury,41,15.65
2,10,Work,31,11.83
3,13,Relationships,26,9.92
4,9,Children,22,8.40
5,0,Other,21,8.02
6,1,Funeral/Death,18,6.87
7,15,Family,18,6.87
8,11,School,10,3.82
9,12,Home/Moving,9,3.44


,topic_label,text_preview
0,Birthday,Two weeks ago I hosted a birthday for Khloe. I...
1,Birthday,I had a good birthday. It was eventful and saw...
2,Birthday,It was during the first weekend of July when I...
3,Birthday,I had the time of my life two weeks ago. I had...
4,Birthday,Just a few months ago I was able to go to my b...
...,...,...
62,Work,One major event in my life is that in the past...
63,Work,I started a company within the past six months...
64,Work,It was a hot day in July that our community de...
65,Work,I retired from work six months ago. I had work...



Full text samples:

[1] topic=Birthday
Two weeks ago I hosted a birthday for Khloe. I had planned this event so that I felt that everything should go well. I thought the day would be easy and successful. I got up early to pick up the final things for the party. I needed to pick up some helium filled balloons. I could not do this ahead of time since the helium would leak out. Next I went to pick up the cake. I wanted it to be as fresh as possible. Having done these last two chores, I returned home. When I unpacked my car, I realized that I had lost my wallet. Oh well, I will call these business concerns later and see if I can locate it. Everyone arrived and things seemed to be going well when out of nowhere Autumn puked. What a mess! I had to clean this up as quickly as possible and make the best of a messy situation. At least the cake was okay. Boy was I glad when this day was over. Everyone seemed to have a good time and now on to find my wallet. I made a few calls and was beginning 

In [41]:
# Verify assigned topic labels against current regex rules
VERIFY_OUTPUT = TARGET_OUTPUT  # set explicitly if needed
CHECK_ROWS = None  # set int to sample N rows for faster checks, e.g., 2000
SHOW_ONLY_MISMATCHES = True
MISMATCH_PREVIEW = 30
VERIFY_TEXT_PREVIEW_WIDTH = 220

verify_path = DATA_DIR / VERIFY_OUTPUT
if not verify_path.exists():
    raise FileNotFoundError(f'Missing file: {verify_path}')

verify_df = pd.read_csv(verify_path)
required_cols = {'topic_label', 'text'}
missing_cols = required_cols.difference(verify_df.columns)
if missing_cols:
    raise ValueError(
        f"{VERIFY_OUTPUT} is missing required columns for verification: {sorted(missing_cols)}"
    )

if CHECK_ROWS is not None and CHECK_ROWS > 0 and len(verify_df) > CHECK_ROWS:
    verify_df = verify_df.sample(CHECK_ROWS, random_state=42).reset_index(drop=True)

def matched_topics(text):
    text_str = str(text)
    return [label for _, label, pattern in RULES if pattern.search(text_str)]

verify_df = verify_df.copy()
verify_df['matched_topics'] = verify_df['text'].apply(matched_topics)
verify_df['first_matched_topic'] = verify_df['matched_topics'].apply(lambda topics: topics[0] if topics else 'Other')
verify_df['n_matched_topics'] = verify_df['matched_topics'].apply(len)
verify_df['assigned_is_first_match'] = verify_df['topic_label'] == verify_df['first_matched_topic']
verify_df['assigned_in_any_match'] = verify_df.apply(
    lambda row: (row['topic_label'] in row['matched_topics']) if row['topic_label'] != 'Other' else (len(row['matched_topics']) == 0),
    axis=1,
 )
verify_df['is_ambiguous'] = verify_df['n_matched_topics'] > 1

summary = (
    verify_df.groupby('topic_label', dropna=False)
    .agg(
        total=('topic_label', 'size'),
        first_match_ok=('assigned_is_first_match', 'sum'),
        any_match_ok=('assigned_in_any_match', 'sum'),
        ambiguous_rows=('is_ambiguous', 'sum'),
    )
    .reset_index()
    .sort_values('total', ascending=False)
    .reset_index(drop=True)
)
summary['first_match_rate_pct'] = (summary['first_match_ok'] / summary['total'] * 100).round(2)
summary['any_match_rate_pct'] = (summary['any_match_ok'] / summary['total'] * 100).round(2)
summary['ambiguous_rate_pct'] = (summary['ambiguous_rows'] / summary['total'] * 100).round(2)

overall_first_match_rate = verify_df['assigned_is_first_match'].mean() * 100 if len(verify_df) else 0.0
overall_ambiguous_rate = verify_df['is_ambiguous'].mean() * 100 if len(verify_df) else 0.0

print(f'Verification dataset: {VERIFY_OUTPUT} | Rows checked: {len(verify_df):,}')
print(f'Overall first-match consistency (meaningful): {overall_first_match_rate:.2f}%')
print(f'Overall multi-hit ambiguity: {overall_ambiguous_rate:.2f}%')
display(summary[['topic_label', 'total', 'first_match_rate_pct', 'any_match_rate_pct', 'ambiguous_rate_pct']])

to_show = verify_df[~verify_df['assigned_is_first_match']].copy()
if SHOW_ONLY_MISMATCHES:
    pass
else:
    to_show = verify_df.copy()

if not to_show.empty:
    from textwrap import shorten
    to_show['text_preview'] = to_show['text'].astype(str).apply(
        lambda s: shorten(s.replace('\n', ' '), width=VERIFY_TEXT_PREVIEW_WIDTH, placeholder=' ...')
    )
    display(
        to_show[['topic_label', 'first_matched_topic', 'n_matched_topics', 'matched_topics', 'text_preview']]
        .head(MISMATCH_PREVIEW)
    )
else:
    print('No first-match mismatches found with current settings.')

Verification dataset: hippocorpus_test_truncated_80_100_labeled.csv | Rows checked: 262
Overall first-match consistency (meaningful): 100.00%
Overall multi-hit ambiguity: 69.08%


,topic_label,total,first_match_rate_pct,any_match_rate_pct,ambiguous_rate_pct
0,Work,43,100.0,100.0,69.77
1,Hospital/Injury,40,100.0,100.0,90.00
2,Vacation/Trip,39,100.0,100.0,76.92
3,Children,20,100.0,100.0,90.00
4,Relationships,20,100.0,100.0,45.00
5,Funeral/Death,18,100.0,100.0,94.44
6,Other,18,100.0,100.0,0.00
7,Money/Gambling,14,100.0,100.0,92.86
8,Family,11,100.0,100.0,0.00
9,Pets,10,100.0,100.0,90.00


No first-match mismatches found with current settings.
